In [2]:
import pandas as pd
import umap
import os
import numpy as np

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import umap.umap_ as umap

/home/wollerf/miniforge3/envs/work/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
base_path = "/home/wollerf/Projects/graph_based_embeddings.git/data/preprocessed/graph_based/"
datasets = ["HANCOCK", "MIMIC", "TCGA_LUAD"]
dimensions = [16, 32, 64]

In [ ]:
for dim in dimensions:
    print("Dimension = ", dim)
    for dataset in datasets:
        print("Dataset = ", dataset)
        for run in range(10):
            file = f"{dataset}_samples_{dim}_{run}.tsv"
            file_path = os.path.join(base_path, dataset, "embeddings", file)
            df = pd.read_csv(file_path, sep='\t', index_col=0)
            # Prepare data.
            X = df.values
            index = df.index  # keep sample IDs
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            
            # PCA visualization.
            pca = PCA(n_components=2, random_state=42)
            X_pca = pca.fit_transform(X_scaled)
            df_pca = pd.DataFrame(
                X_pca,
                index=index,
                columns=["PCA_1", "PCA_2"]
            )
            
            # UMAP visualization.
            umap_model = umap.UMAP(
                n_components=2,
                metric="euclidean",
                random_state=42
            )
            X_umap = umap_model.fit_transform(X_scaled)
            df_umap = pd.DataFrame(
                X_umap,
                index=index,
                columns=["UMAP_1", "UMAP_2"]
            )

            # t-SNE visualization.
            tsne = TSNE(
                n_components=2,
                init="pca",
                random_state=42
            )
            X_tsne = tsne.fit_transform(X_scaled)
            df_tsne = pd.DataFrame(
                X_tsne,
                index=index,
                columns=["tSNE_1", "tSNE_2"]
            )
            
            # Combine into one df and save.
            df_embeddings = pd.concat(
                [df_pca, df_umap, df_tsne],
                axis=1
            )
            out_path = os.path.join(base_path, dataset, "visualization", file)
            df_embeddings.to_csv(out_path, sep='\t', index=True)
            
            